In [ ]:
!pip install transformers datasets evaluate

import numpy as np
import torch
from torch.optim import AdamW
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments, TrainerCallback, set_seed

import evaluate
from datasets import load_dataset
from transformers import EarlyStoppingCallback

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

In [ ]:
SEED = 42
set_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'


In [ ]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2).to(device)
model.config.id2label = {0: 'NEGATIVE', 1: 'POSITIVE'}
model.config.label2id = {'NEGATIVE': 0, 'POSITIVE': 1}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
dataset = load_dataset('glue', 'sst2')


README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

In [ ]:
max_length = 128
def tokenize_fn(example):
    enc = tokenizer(
        example['sentence'],
        padding=True,
        truncation=True,
        max_length=max_length
    )
    return {
        'input_ids': enc['input_ids'],
        'attention_mask': enc['attention_mask'],
        'labels': example['label']
    }

dataset = dataset.map(tokenize_fn, batched=True, batch_size=2000)
dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])


Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

In [ ]:
train_data = dataset['train'].shuffle(seed=SEED).select(range(5000))  # small subset for quick runs
val_data = dataset['validation']


In [ ]:
accuracy = evaluate.load('accuracy')
f1 = evaluate.load('f1')
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc_result = accuracy.compute(predictions=predictions, references=labels)
    f1_result = f1.compute(predictions=predictions, references=labels, average='weighted')
    return {**acc_result, **f1_result}


In [ ]:
#freeze/unfreeze functions
def freeze_layers(model, num_freeze=6):
    for name, param in model.bert.encoder.layer[:num_freeze].named_parameters():
        param.requires_grad = False

def unfreeze_all(model):
    for param in model.parameters():
        param.requires_grad = True

class FreezeUnfreezeCallback(TrainerCallback):
    def on_epoch_begin(self, args, state, control, model=None, **kwargs):
        if state.epoch < 1:
            freeze_layers(model, num_freeze=6)
        else:
            unfreeze_all(model)
        return control


In [ ]:

training_args = TrainingArguments(
    output_dir='./bert-sst2-fine-tuned',
    seed=SEED,
    data_seed=SEED,
    num_train_epochs=2,
    gradient_accumulation_steps=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
    report_to='none',
    fp16=True,
    save_total_limit=1,
    greater_is_better=True,
    metric_for_best_model='accuracy',
    load_best_model_at_end=True
)


In [ ]:

#earlyStopping
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[
        FreezeUnfreezeCallback(),
        EarlyStoppingCallback(early_stopping_patience=2)
    ],
    optimizers=(None, None)
)


In [ ]:
trainer.train()

final_eval = trainer.evaluate()
print('Final Evaluation : \n')
print(f"Eval Loss: {final_eval['eval_loss']:.4f} | Eval Accuracy: {final_eval['eval_accuracy']:.4f} | Eval F1 Score: {final_eval['eval_f1']:.4f}")

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.302900,0.289506,0.893349,0.893317


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
